# 00 — Database Setup
---
**Run this notebook once before anything else in the project.**

This notebook:
1. Unzips the raw NHL data files from 
2. Moves all CSV files into a flat structure under 
3. Loads each CSV into a SQLite database at 
4. Verifies every table loaded correctly with row counts and column previews

**Data source:** [NHL Game Data on Kaggle](https://www.kaggle.com/datasets/martinellis/nhl-game-data)

---

## 0. Environment Setup

In [2]:
import sys
from pathlib import Path

# Add project root to path so we can import config and utils
ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

from config import DB_PATH, RAW_DIR
from utils.file_utils import unzip_all_files, move_all_files
from utils.db_connection import create_database_from_csvs, list_tables, table_info, run_query

print(f"Project root : {ROOT}")
print(f"Raw data dir : {RAW_DIR}")
print(f"Database path: {DB_PATH}")

Project root : c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics
Raw data dir : c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\raw
Database path: c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\NHL_data.db


---
## Step 1 — Unzip Raw Data Files

Extracts all  files found in .
Each zip extracts into a subfolder — Step 2 will flatten those into .

In [12]:
unzip_all_files(RAW_DIR)

Extracting: nhl_data_files_1.zip
Extracting: nhl_data_files_2.zip
Done extracting.


---
## Step 2 — Flatten CSV Files into 

The zips extract into subdirectories (, , etc.).
This step moves all CSV files up into  so they are easy to find and load.

In [13]:
# Move files from each extracted subdirectory into data/raw/
# If a folder does not exist it will be skipped gracefully
subdirs_to_flatten = [
    "nhl_data_files_1",
    "nhl_data_files_2"
]

for subdir in subdirs_to_flatten:
    source = RAW_DIR / subdir
    if source.exists():
        move_all_files(source, RAW_DIR)
    else:
        print(f"  -  {subdir}/ not found, skipping.")

Moved: game_shifts.csv → c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\raw
Moved: game_skater_stats.csv → c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\raw
Moved: game_teams_stats.csv → c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\raw
Moved: player_info.csv → c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\raw
Moved: team_info.csv → c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\raw
Done moving files.
Moved: game.csv → c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\raw
Moved: game_goals.csv → c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\raw
Moved: game_penalties.csv → c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\raw
Moved: game_plays_players.csv → c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\raw
Moved: game_plays_trimmed.csv → c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-an

---
## Step 3 — Preview Available CSV Files

Confirm all expected files are present before building the database.

In [3]:
# The 9 standard CSV files — loaded as-is into the database
# game_plays.csv is handled separately below due to its size and noise filtering
EXPECTED_FILES = [
    'game.csv',
    'game_goals.csv',
    'game_penalties.csv',
    'game_plays_players.csv',
    'game_shifts.csv',
    'game_skater_stats.csv',
    'game_teams_stats.csv',
    'player_info.csv',
    'team_info.csv',
]

print('Checking for expected CSV files in data/raw/:\n')
all_present = True
for fname in EXPECTED_FILES:
    fpath = RAW_DIR / fname
    status = 'OK     ' if fpath.exists() else 'MISSING'
    print(f'  {status}  {fname}')
    if not fpath.exists():
        all_present = False

# Also check for the big one
gp = RAW_DIR / 'game_plays.csv'
gp_status = 'OK     ' if gp.exists() else 'MISSING'
print(f'  {gp_status}  game_plays.csv  (processed separately)')
if not gp.exists():
    all_present = False

print()
if all_present:
    print('All expected files present. Ready to build database.')
else:
    print('WARNING: Some files are missing. Check zip contents before continuing.')

Checking for expected CSV files in data/raw/:

  OK       game.csv
  OK       game_goals.csv
  OK       game_penalties.csv
  OK       game_plays_players.csv
  OK       game_shifts.csv
  OK       game_skater_stats.csv
  OK       game_teams_stats.csv
  OK       player_info.csv
  OK       team_info.csv
  OK       game_plays.csv  (processed separately)

All expected files present. Ready to build database.


---
## Step 4 — Build the SQLite Database

Loads each CSV file into  as its own table.
Table names match the CSV filenames (e.g.  becomes table ).

Note: This takes 1-2 minutes.  and  are the largest files.

In [4]:
# Load the 9 standard CSV files into the database
csv_paths = [RAW_DIR / fname for fname in EXPECTED_FILES]
create_database_from_csvs(csv_paths, DB_PATH)

Building database at: c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\NHL_data.db

  OK  game                               26,305 rows
  OK  game_goals                        148,992 rows
  OK  game_penalties                    247,828 rows
  OK  game_plays_players              7,586,604 rows
  OK  game_shifts                    11,882,735 rows
  OK  game_skater_stats                 945,830 rows
  OK  game_teams_stats                   52,610 rows
  OK  player_info                         3,925 rows
  OK  team_info                              33 rows

Done. Database saved to c:\Users\abonc\OneDrive\Documents\Ailene\Projects\nhl-analytics\data\NHL_data.db


---
## Step 4b — Load `game_plays.csv` with Smart Filtering

The original `game_plays.csv` has **5 million rows** spanning 2000–2020,
but roughly half are administrative noise events with no analytical value
(`Game Scheduled`, `Period Ready`, `Period Start`, etc.).

Rather than loading all 5M rows or filtering by date like the original project did,
we filter by **event type** — keeping only the events that matter for analysis
while retaining all seasons. This gives us full historical coverage with a
lean, query-fast table.

The resulting table is named **`game_plays`** in the database.

In [5]:
import pandas as pd
import sqlite3

# Events worth keeping — everything with analytical signal
KEEP_EVENTS = {
    'Shot', 'Goal', 'Hit', 'Faceoff', 'Penalty',
    'Missed Shot', 'Blocked Shot', 'Giveaway', 'Takeaway'
}

# Columns to drop — st_x/st_y duplicate x/y; description is free text we don't need
DROP_COLS = ['st_x', 'st_y', 'description']

print('Reading game_plays.csv (this may take a moment)...')
gp = pd.read_csv(RAW_DIR / 'game_plays.csv', low_memory=False)
print(f'  Original shape: {gp.shape[0]:,} rows x {gp.shape[1]} cols')

# Filter to meaningful events only
gp = gp[gp['event'].isin(KEEP_EVENTS)]
print(f'  After event filter: {gp.shape[0]:,} rows')

# Drop redundant/unneeded columns
gp = gp.drop(columns=[c for c in DROP_COLS if c in gp.columns])
print(f'  Final shape: {gp.shape[0]:,} rows x {gp.shape[1]} cols')

# Load into database
print('\nLoading into database as table: game_plays')
conn = sqlite3.connect(DB_PATH)
gp.to_sql('game_plays', conn, if_exists='replace', index=False)
conn.commit()
conn.close()
print(f'  Done. {gp.shape[0]:,} rows loaded into game_plays.')

Reading game_plays.csv (this may take a moment)...
  Original shape: 5,050,529 rows x 18 cols
  After event filter: 4,116,425 rows
  Final shape: 4,116,425 rows x 15 cols

Loading into database as table: game_plays
  Done. 4,116,425 rows loaded into game_plays.


---
## Step 5 — Verify the Database

Four checks to confirm everything loaded correctly.

In [7]:
# Check 1: Table names
tables = list_tables()
print(f"Tables in database ({len(tables)} total):"
)
for t in tables:
    print(f"  - {t}")

Tables in database (10 total):
  - game
  - game_goals
  - game_penalties
  - game_plays
  - game_plays_players
  - game_shifts
  - game_skater_stats
  - game_teams_stats
  - player_info
  - team_info


In [8]:
# Check 2: Row counts for every table
print(f"{'Table':<30} {'Row Count':>12}")
print('-' * 44)
for t in tables:
    count = run_query(f'SELECT COUNT(*) as n FROM {t}')['n'].iloc[0]
    flag = '  <-- WARNING: empty\!' if count == 0 else ''
    print(f'  {t:<28} {count:>12,}{flag}')

<>:6: SyntaxWarning: "\!" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\!"? A raw string is also an option.
<>:6: SyntaxWarning: "\!" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\!"? A raw string is also an option.
C:\Users\abonc\AppData\Local\Temp\ipykernel_28980\1015337598.py:6: SyntaxWarning: "\!" is an invalid escape sequence. Such sequences will not work in the future. Did you mean "\\!"? A raw string is also an option.
  flag = '  <-- WARNING: empty\!' if count == 0 else ''


Table                             Row Count
--------------------------------------------
  game                               26,305
  game_goals                        148,992
  game_penalties                    247,828
  game_plays                      4,116,425
  game_plays_players              7,586,604
  game_shifts                    11,882,735
  game_skater_stats                 945,830
  game_teams_stats                   52,610
  player_info                         3,925
  team_info                              33


In [10]:
# Check 3: Column structure for key tables
key_tables = ["game", "game_skater_stats", "player_info", "team_info"]

for t in key_tables:
    print(f"-- {t} --")
    info = table_info(t)
    print(info[["name", "type"]].to_string(index=False))

-- game --
                  name    type
               game_id INTEGER
                season INTEGER
                  type    TEXT
         date_time_GMT    TEXT
          away_team_id INTEGER
          home_team_id INTEGER
            away_goals INTEGER
            home_goals INTEGER
               outcome    TEXT
  home_rink_side_start    TEXT
                 venue    TEXT
            venue_link    TEXT
    venue_time_zone_id    TEXT
venue_time_zone_offset INTEGER
    venue_time_zone_tz    TEXT
-- game_skater_stats --
                name    type
             game_id INTEGER
           player_id INTEGER
             team_id INTEGER
           timeOnIce INTEGER
             assists INTEGER
               goals INTEGER
               shots INTEGER
                hits    REAL
      powerPlayGoals INTEGER
    powerPlayAssists INTEGER
      penaltyMinutes INTEGER
         faceOffWins INTEGER
        faceoffTaken INTEGER
           takeaways    REAL
           giveaways    REAL
    s

In [11]:
# Check 4: Seasons and game types available in the game table
seasons = run_query("""
    SELECT season, type, COUNT(*) as games
    FROM game
    GROUP BY season, type
    ORDER BY season, type
""")

print("Seasons and game types available in the database:")
print(seasons.to_string(index=False))

Seasons and game types available in the database:
  season type  games
20002001    R   1230
20012002    R   1230
20022003    R   1230
20032004    R   1230
20052006    R   1230
20062007    R   1230
20072008    R   1230
20082009    R   1230
20092010    R   1230
20102011    P     89
20102011    R   1230
20112012    P     86
20112012    R   1230
20122013    P     86
20122013    R    720
20132014    P     93
20132014    R   1230
20142015    P     89
20142015    R   1230
20152016    P     91
20152016    R   1230
20162017    P     93
20162017    R   1230
20172018    P     92
20172018    R   1271
20182019    A      4
20182019    P    174
20182019    R   2542
20192020    A      6
20192020    P    231
20192020    R   2188


---
## Setup Complete

If all tables show row counts above and the seasons query returned results, the database is ready.

**Next:** Open  to begin exploring the full dataset.